In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer, models

# 1. Initialize the Ancient Greek BERT Model
# We add a Mean Pooling layer to convert word tokens into a single verse vector
word_embedding_model = models.Transformer("models/ancient-greek-biblical-sbert")
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(), 
    pooling_mode='mean'
)
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# 2. Generate Embeddings (This will take a few minutes for 38k verses)
print("Generating embeddings...")
ot_nt_final_df = pd.read_csv("bible_corpus.csv")
sentences = ot_nt_final_df['text'].tolist()
embeddings = model.encode(sentences, show_progress_bar=True, convert_to_numpy=True)

# 3. Normalize for Cosine Similarity
# Normalizing vectors to unit length allows us to use Inner Product (IP) for Cosine Similarity
faiss.normalize_L2(embeddings)

# 4. Create and Build the FAISS Index
dimension = embeddings.shape[1] # Usually 768 for BERT
index = faiss.IndexFlatIP(dimension) 
index.add(embeddings)

# 5. Save Everything to Disk
# We save the FAISS index and the DataFrame separately to reload later
faiss.write_index(index, "bible_greek.index")
ot_nt_final_df.to_pickle("bible_metadata.pkl")

print(f"Database created with {index.ntotal} vectors.")

# --- Quick Test Function ---
def semantic_search(query, top_k=5):
    # Encode and normalize the search term
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)
    
    # Search the index
    distances, indices = index.search(query_vector, top_k)
    
    # Return results
    return ot_nt_final_df.iloc[indices[0]]


Loading weights:   9%|▊         | 17/199 [00:00<00:00, 1295.50it/s, Materializing param=encoder.layer.0.intermediate.dense.weight]       

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2756.79it/s, Materializing param=pooler.dense.weight]                               


Generating embeddings...


Batches: 100%|██████████| 1204/1204 [00:49<00:00, 24.33it/s]


Database created with 38526 vectors.


In [5]:

# Usage:
results = semantic_search("ἀγάπη")
print(results)

                verse_ref                                               text
476            1 john 3:1  ἴδετε ποταπὴν ἀγάπην δέδωκεν ἡμῖν ὁ πατήρ, ἵνα...
426   1 corinthians 16:14                       πάντα ὑμῶν ἐν ἀγάπῃ γινέσθω.
3808           john 15:13  μείζονα ταύτης ἀγάπην οὐδεὶς ἔχει ἵνα τις τὴν ...
621           1 peter 4:8  πρὸ πάντων δὲ τὴν εἰς ἑαυτοὺς ἀγάπην ἐκτενῆ ἔχ...
517           1 john 4:18  φόβος οὐκ ἔστιν ἐν τῇ ἀγάπῃ, ἀλλ᾽ ἡ τελεία ἀγ...
